In [1]:
import json
import os
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

os.environ["USE_TF"] = "0"

print("PyTorch versión:", torch.__version__)

C:\Users\udaet\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch versión: 2.15.0+cu132


In [3]:
# 1. Cargar base de datos documental
with open("data/pokemon_data.json", "r", encoding="utf-8") as f:
    documentos = json.load(f)

print(f"Documentos cargados: {len(documentos)}")

# 2. Cargar modelos RAG
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")
generador = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=-1
)

print("Modelos cargados con éxito.")

Documentos cargados: 1


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2325.33it/s]
C:\Users\udaet\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\udaet\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-

Modelos cargados con éxito.


In [4]:
textos_documentos = [doc["texto"] for doc in documentos]

embeddings_documentos = modelo_embeddings.encode(
    textos_documentos,
    convert_to_numpy=True
)

print("Forma de la matriz de embeddings:", embeddings_documentos.shape)

Forma de la matriz de embeddings: (1, 384)


In [5]:
def recuperar_documentos(pregunta, k=2):
    embedding_pregunta = modelo_embeddings.encode(pregunta, convert_to_numpy=True)
    puntuaciones = cosine_similarity([embedding_pregunta], embeddings_documentos)[0]
    indices = np.argsort(puntuaciones)[::-1][:k]

    resultados = []
    for indice in indices:
        resultados.append({
            "documento": documentos[indice],
            "puntuacion": float(puntuaciones[indice])
        })
    return resultados

def construir_contexto(resultados):
    partes = []
    for res in resultados:
        doc = res["documento"]
        partes.append(f"[Documento {doc['id']} | Fuente: {doc['fuente']}]\n{doc['texto']}")
    return "\n\n".join(partes)

def generar_respuesta(pregunta, contexto):
    prompt = f'''
    Responde a la pregunta utilizando únicamente la información del contexto.

    Si el contexto no contiene la respuesta, indica:
    "No encuentro esa información en los documentos disponibles."

    No inventes datos ni añadas información externa.

    CONTEXTO:
    {contexto}

    PREGUNTA:
    {pregunta}

    RESPUESTA:
    '''

    salida = generador(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )
    return salida[0]["generated_text"].strip()

def consultar_rag(pregunta, k=2):
    resultados = recuperar_documentos(pregunta, k=k)
    contexto = construir_contexto(resultados)
    respuesta = generar_respuesta(pregunta, contexto)

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "documentos_recuperados": resultados
    }

In [6]:
# Prueba 1: Consulta existente
res1 = consultar_rag("Acabo de capturar un Charmander, ¿a qué nivel evoluciona?")
print("PREGUNTA:", res1["pregunta"])
print("RESPUESTA:", res1["respuesta"])
print("DOCUMENTOS:", [i["documento"]["id"] for i in res1["documentos_recuperados"]])
print("=" * 60)

# Prueba 2: Consulta fuera de la base documental
res2 = consultar_rag("¿A qué nivel evoluciona Mewtwo?")
print("PREGUNTA:", res2["pregunta"])
print("RESPUESTA:", res2["respuesta"])
print("DOCUMENTOS:", [i["documento"]["id"] for i in res2["documentos_recuperados"]])

PREGUNTA: Acabo de capturar un Charmander, ¿a qué nivel evoluciona?
RESPUESTA: Responde a la pregunta utilizando únicamente la información del contexto.

    Si el contexto no contiene la respuesta, indica:
    "No encuentro esa información en los documentos disponibles."

    No inventes datos ni añadas información externa.

    CONTEXTO:
    [Documento 4 | Fuente: pokeapi.co]
Charmander es un Pokémon de tipo Fuego. Evolución: Charmander evoluciona a Charmeleon al nivel 16. Charmeleon evoluciona a Charizard al nivel 36. Stats clave: Atq. Esp: 60, Atq: 52, Vel: 65, Def: 43. Destaca en Ataque Especial. Movimientos destacados: Mega punch, Fire punch, Thunder punch, Scratch, Swords dance, Cut, Bind, Vine whip.

    PREGUNTA:
    Acabo de capturar un Charmander, ¿a qué nivel evoluciona?

    RESPUESTA:
     16

En este problema se requiere determinar el nivel de evolución de un Pokémon dado su nombre y sus características específicas. El contexto proporcionado incluye información sobre el 